# expdpy — Quickstart

A cloud-runnable walkthrough of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

This notebook mirrors the [Quickstart page](https://cmg777.github.io/expdpy/quickstart.html) of the docs.

In [ ]:
!pip install -q "git+https://github.com/cmg777/expdpy.git"

This walkthrough mirrors the *ExPanDaR functions* vignette using the bundled `kuznets`
panel (80 countries × 2015–2025) — a synthetic dataset, rich in control variables, whose
regional inequality (`gini_regional`) traces an **N-shaped Kuznets curve** in income: it
rises, falls, then rises again at very high income. The log of GDP per capita and its square
and cube ship as columns (`log_gdp_pc`, `log_gdp_pc_sq`, `log_gdp_pc_cu`) so the cubic curve
is turnkey.

In [ ]:
import numpy as np
import expdpy as ex
from expdpy.data import load_kuznets

df = load_kuznets()
df[["country", "continent", "year", "gini_regional", "gdp_pc", "log_gdp_pc"]].head()

## Treating outliers

Winsorize the (skewed) numeric variables at the 1st/99th percentile:

In [ ]:
analysis = ex.treat_outliers(
    df[["gini_regional", "gdp_pc", "school_enrollment", "resource_rents"]], percentile=0.01
).dropna()
analysis.describe().round(3)

## Descriptive statistics

In [ ]:
ex.prepare_descriptive_table(analysis).gt

## Correlations

Pearson correlations appear above, Spearman below the diagonal; significant cells are bold.

In [ ]:
ex.prepare_correlation_table(analysis).gt

In [ ]:
ex.prepare_correlation_graph(analysis).fig

## Extreme observations

In [ ]:
ex.prepare_ext_obs_table(
    df, n=5, cs_id=["country"], ts_id="year", var="gini_regional"
).gt

## Time trends

In [ ]:
ex.prepare_trend_graph(df, ts_id="year", var=["gini_regional"]).fig

In [ ]:
ex.prepare_quantile_trend_graph(df, ts_id="year", var="gini_regional").fig

## By-group views

In [ ]:
ex.prepare_by_group_bar_graph(df, "continent", "gini_regional", np.nanmedian).fig

In [ ]:
ex.prepare_by_group_violin_graph(df, "continent", "gini_regional")

## Scatter plot with LOESS

The **N-shaped Kuznets curve** — regional inequality against (log) GDP per capita, sized by
population and colored by continent. The LOESS smoother traces the rise–fall–rise:

In [ ]:
ex.prepare_scatter_plot(
    df, x="log_gdp_pc", y="gini_regional", color="continent", size="population", loess=1
)

## Regression with fixed effects and clustered SEs

`kuznets` is a country–year panel, so the natural specification absorbs **two-way (country +
year) fixed effects** — controlling for every time-invariant country trait and every common
annual shock. A cubic in (log) GDP per capita still recovers the N — a positive, significant
cubic term — *within* country, with standard errors clustered by country:

In [ ]:
res = ex.prepare_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
)
res.etable

## Frisch-Waugh-Lovell plot

The same coefficient, *seen*. `prepare_fwl_plot` residualizes both the outcome and the focal
regressor on the **other** regressors **and** the fixed effects, then scatters the two
residuals. By the Frisch-Waugh-Lovell theorem the fitted slope equals the focal coefficient
in the regression above — here the linear `log_gdp_pc` term, net of the quadratic/cubic terms
and the country and year fixed effects:

In [ ]:
ex.prepare_fwl_plot(
    df,
    dv="gini_regional",
    var="log_gdp_pc",
    controls=["log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
).fig